In [1]:
"""
This script is used to calculate normalized betas of atac to rna per sample
authors: Roy Oelen
"""


'\nThis script is used to calculate normalized betas of atac to rna per sample\nauthors: Roy Oelen\n'

In [2]:
#############
# libraries #
#############

# standard format for CPU
import numpy as np
# standard format for GPU
import cupy as cp
# we need these matrices, coo to get the matrix, and csr to actually use
from scipy.sparse import csr_matrix
from scipy.sparse import coo_matrix
# we read the mtx.gz files
from scipy.io import mmread
# to test if we are dealing with a scipy or cupy implementation
from scipy.sparse import issparse
# the cupy implementation of the matrix
from cupyx.scipy.sparse import csr_matrix as cp_csr_matrix
from cupyx.scipy.sparse import coo_matrix as cp_coo_matrix
# to test if we are dealing with a scipy or cupy implementation
from cupyx.scipy.sparse import issparse as cp_issparse
# we use the scipy versions of these tools in CPU-only scenarios
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler
from scipy.stats import yeojohnson
# files might be zipped
import gzip

In [3]:
#############
# functions #
#############

def load_matrix_cpu(matrix_loc):
    """
    Load a sparse matrix from a .mtx.gz file using SciPy.

    Parameters:
    matrix_loc (str): The file path to the .mtx.gz file.

    Returns:
    scipy.sparse.coo_matrix: The loaded sparse matrix in COO format.
    """
    # load the matrix as coo
    with gzip.open(matrix_loc, 'rt') as f:
        scipy_sparse_matrix = mmread(f)
    return scipy_sparse_matrix


def load_matrix_gpu(matrix_loc):
    """
    Load a sparse matrix from a .mtx.gz file using SciPy and convert it to a CuPy sparse matrix.

    Parameters:
    matrix_loc (str): The file path to the .mtx.gz file.

    Returns:
    cupyx.scipy.sparse.csr_matrix: The loaded sparse matrix in CSR format.
    """
    # load using scipy cpu function
    scipy_sparse_matrix = load_matrix_cpu(matrix_loc)
    # convert the data type to be compatible with CuPy
    scipy_sparse_matrix.data = scipy_sparse_matrix.data.astype(cp.float64)
    # convert the SciPy coo sparse matrix to a CuPy sparse matrix
    cupy_sparse_matrix = cp_csr_matrix(scipy_sparse_matrix)
    # clear memory
    del scipy_sparse_matrix
    return cupy_sparse_matrix


def load_matrix(matrix_loc):
    """
    Load a sparse matrix from a .mtx.gz file using either CPU or GPU implementation.

    Parameters:
    matrix_loc (str): The file path to the .mtx.gz file.

    Returns:
    scipy.sparse.csr_matrix or cupyx.scipy.sparse.csr_matrix: The loaded sparse matrix in CSR format.
    """
    # use GPU or CPU implementation
    if use_gpu:
        return load_matrix_gpu(matrix_loc)
    else:
        return load_matrix_cpu(matrix_loc).tocsr()

    
def yeo_johnson_normalization_and_scale_row_cpu(row):
    """
    Apply Yeo-Johnson normalization and scaling to a single row using CPU.

    Parameters:
    row (scipy.sparse.csr_matrix): A single row of the sparse matrix.

    Returns:
    numpy.ndarray: The normalized and scaled row.
    """
    # init transformer object
    transformer = PowerTransformer(method='yeo-johnson', standardize=True)
    # init scaler object
    scaler = StandardScaler()
    # convert sparse row to dense format
    row_dense = row.toarray().flatten()
    # use the transformer
    row_transformed = transformer.fit_transform(row_dense.reshape(-1, 1)).flatten()
    # use the scaler
    row_scaled = scaler.fit_transform(row_transformed.reshape(-1, 1)).flatten()
    return row_scaled


def yeo_johnson_normalization_and_scale_cpu(matrix):
    """
    Apply Yeo-Johnson normalization and scaling to each row of the matrix using CPU.

    Parameters:
    matrix (scipy.sparse.csr_matrix): The sparse matrix.

    Returns:
    numpy.ndarray: The normalized and scaled matrix.
    """
    # setup threadpool
    with ProcessPoolExecutor() as executor:
        # do row per thread
        normalized_rows = list(executor.map(yeo_johnson_normalization_and_scale_row_cpu, matrix))
    # put restult in numpy array
    return np.array(normalized_rows)


def yeo_johnson_transform(x, lmbda):
    """
    Apply the Yeo-Johnson transformation to a CuPy array.

    Parameters:
    x (cupy.ndarray): The input array.
    lmbda (float): The lambda parameter for the Yeo-Johnson transformation.

    Returns:
    cupy.ndarray: The transformed array.
    """
    """Apply the Yeo-Johnson transformation to a CuPy array."""
    
    # create output array that is the same shape as x, but with zeroes
    out = cp.zeros_like(x)
    # create booleans of the positive (or zero) and negative values
    pos = x >= 0
    neg = ~pos

    # if the supplied lambda is 0, the transformation for non-negative values is the natural logarithm of x + 1
    if lmbda == 0:
        out[pos] = cp.log1p(x[pos])
    # otherwise it is a bit more complicated
    else:
        out[pos] = (cp.power(x[pos] + 1, lmbda) - 1) / lmbda
    # if the lambda is 2, the transformation for negative values is the negative natural logarithm of 1 - x
    if lmbda == 2:
        out[neg] = -cp.log1p(-x[neg])
    # otherwise it is again a bit more complicated
    else:
        out[neg] = -((cp.power(-x[neg] + 1, 2 - lmbda) - 1) / (2 - lmbda))
    
    return out


def handle_nan_inf(matrix):
    """
    Replace NaN and infinite values in the matrix with zeros.

    Parameters:
    matrix (cupyx.scipy.sparse.csr_matrix): The sparse matrix.

    Returns:
    cupyx.scipy.sparse.csr_matrix: The matrix with NaN and infinite values replaced by zeros.
    """
    # make all NAN zero
    matrix.data[cp.isnan(matrix.data)] = 0
    # make all infinite zero
    matrix.data[cp.isinf(matrix.data)] = 0
    return matrix


def yeo_johnson_normalization_and_scale_row_gpu(row):
    """
    Apply Yeo-Johnson normalization and scaling to a single row using GPU.

    Parameters:
    row (cupyx.scipy.sparse.csr_matrix): A single row of the sparse matrix.

    Returns:
    cupy.ndarray: The normalized and scaled row.
    """
    # make the row into dense format
    row_dense = row.toarray().flatten()
    
    # find the optimal lambda for the Yeo-Johnson transformation
    _, lmbda = yeojohnson(row_dense.get())
    
    # apply the Yeo-Johnson transformation using the optimal lambda we determined
    row_transformed = yeo_johnson_transform(cp.array(row_dense), lmbda)
    
    # Scale the transformed row using CuPy, same implementation as StandardScaler of scipy
    row_mean = cp.mean(row_transformed)
    row_std = cp.std(row_transformed) + 1e-8  # Add epsilon to avoid division by zero
    row_scaled = (row_transformed - row_mean) / row_std
    
    return row_scaled


def yeo_johnson_normalization_and_scale_gpu(cp_matrix):
    """
    Apply Yeo-Johnson normalization and scaling to each row of the matrix using GPU.

    Parameters:
    cp_matrix (cupyx.scipy.sparse.csr_matrix): The sparse matrix.

    Returns:
    cupy.ndarray: The normalized and scaled matrix.
    """
    # do each row, cupy should leverage paralellism automatically
    normalized_rows = cp.array([yeo_johnson_normalization_and_scale_row_gpu(row) for row in cp_matrix])
    return normalized_rows


def yeo_johnson_normalization_and_scale(matrix):
    """
    Apply Yeo-Johnson normalization and scaling to each row of the matrix using either CPU or GPU.

    Parameters:
    matrix (scipy.sparse.csr_matrix or cupyx.scipy.sparse.csr_matrix): The sparse matrix.

    Returns:
    numpy.ndarray or cupy.ndarray: The normalized and scaled matrix.
    """
    # depending on whether we have a GPU, do the CPU or GPU implementation
    if use_gpu:
        return yeo_johnson_normalization_and_scale_gpu(matrix)
    else:
        return yeo_johnson_normalization_and_scale_cpu(matrix)

In [4]:
# whether we use GPU or not
use_gpu = True

In [5]:
# location of the RNA matrix
rna_matrix_loc = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/qtl/coeqtl/trial_run/matrices/DC/MO100_230202_lane6/RNA/matrix.mtx.gz'
# location of the ATAC matrix
atac_matrix_loc = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/qtl/coeqtl/trial_run/matrices/DC/MO100_230202_lane6/peaks/matrix.mtx.gz'

In [6]:
# load the two matrices
rna_matrix = load_matrix(rna_matrix_loc)
atac_matrix = load_matrix(atac_matrix_loc)

In [7]:
# do yeo johnson normalization on the rna data
rna_normal = yeo_johnson_normalization_and_scale(rna_matrix)

In [ ]:
# location of the CREs to test
cre_loc = ''